In [ ]:
import numpy as np
import torch
import torch.optim as optim
import logging
import matplotlib.pyplot as plt
from argparse import ArgumentParser
from torch.autograd import Variable
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.distributions import Normal, OneHotCategorical
#from bayes_opt import BayesianOptimization
import torch.nn.utils as nn_utils

torch.manual_seed(123456)
np.random.seed(123456)


def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_normal_(m.weight)
        nn.init.constant_(m.bias, 0.0)
        
class MDN(nn.Module):
    def __init__(self, n_hidden, n_gaussians,n_hidden_layers,bn, dout, dout_value):
        super(MDN, self).__init__()
        
        
        layers = [nn.Linear(41, n_hidden), nn.Tanh()]
        if bn == 1:
            layers.append(nn.BatchNorm1d(n_hidden))    
        for _ in range(n_hidden_layers - 2):
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.Tanh())
        if dout==1:
            layers.append(nn.Dropout(dout_value))
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.Tanh())   
        else:
            layers.append(nn.Linear(n_hidden, n_hidden))
            layers.append(nn.Tanh())       
        self.z_h = nn.Sequential(*layers)
        

        
        
        self.z_pi = nn.Linear(n_hidden, n_gaussians)
        self.z_sigma = nn.Linear(n_hidden, n_gaussians)
        self.z_mu = nn.Linear(n_hidden, n_gaussians)  

    def forward(self, x):
        z_h = self.z_h(x)
        pi = nn.functional.softmax(self.z_pi(z_h), -1)
        sigma = torch.exp(self.z_sigma(z_h))+ 1e-8
        sigma = torch.clamp(sigma, min=1e-4)
        mu = 0 + (1 - 0) * (torch.tanh(self.z_mu(z_h)) + 1) / 2
        return pi, sigma, mu
    
oneDivSqrtTwoPI = 1.0 / np.sqrt(2.0*np.pi) 
def gaussian_distribution(y, mu, sigma):
    result = (y.expand_as(mu) - mu) * torch.reciprocal(sigma)
    result = -0.5 * (result * result)
    return (torch.exp(result) * torch.reciprocal(sigma)) * oneDivSqrtTwoPI

def mdn_loss_fn(pi, sigma, mu, y):
    result = gaussian_distribution(y, mu, sigma) * pi
    result1 = torch.sum(result, dim=1)
    result2 = -torch.log(result1+1e-12)
    return torch.mean(result2)



def nn_cl_bo2(dropout, dropout_rate, normalization, batch_size, layers1, neurons, learning_rate): 
    n_epochs = 1000
    
    kfold = 10
    num_gaussians = 5  
    rng = 0
    device = 'cpu'
    
    layers1 = round(layers1)
    neurons = round(neurons)
    batch_size = round(batch_size)
    
    bnorm = 0
    if normalization > 0.5:
        bnorm = 1
    dout = 0
    if dropout > 0.5:
        dout = 1
        
    xtrain=np.load('xlo_oversampling.npy')
    ytrain=np.load('ylo_oversampling.npy')
    x0test=np.load('xlo_val_0.npy')
    y0test=np.load('ylo_val_0.npy')[:,1:2]
    x1test=np.load('xlo_val_1.npy')
    y1test=np.load('ylo_val_1.npy')[:,1:2]
    x2test=np.load('xlo_val_2.npy')
    y2test=np.load('ylo_val_2.npy')[:,1:2]
    x3test=np.load('xlo_val_3.npy')
    y3test=np.load('ylo_val_3.npy')[:,1:2]
    xtest=np.vstack([x0test,x1test,x2test,x3test])
    ytest=np.vstack([y0test,y1test,y2test,y3test])
    x0=np.load('xlo_0.npy')
    y0=np.load('ylo_0.npy')[:,1:2]
    x1=np.load('xlo_1.npy')
    y1=np.load('ylo_1.npy')[:,1:2]
    x2=np.load('xlo_2.npy')
    y2=np.load('ylo_2.npy')[:,1:2]
    x3=np.load('xlo_3.npy')
    y3=np.load('ylo_3.npy')[:,1:2]
    xtrain=np.vstack([xtrain,x0,x1,x2,x3])
    ytrain=np.vstack([ytrain,y0,y1,y2,y3])
    num_rows=np.shape(xtrain)[0]
    permutation = np.random.permutation(num_rows)
    xtrain = xtrain[permutation]
    ytrain = ytrain[permutation]
    

    x_train = torch.tensor(xtrain, dtype=torch.float)
    y_train = torch.tensor(ytrain, dtype=torch.float)
    x_test = torch.tensor(xtest, dtype=torch.float)
    y_test = torch.tensor(ytest, dtype=torch.float)
    test_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(x_test, y_test),
        batch_size=x_test.shape[0], shuffle=False, drop_last=True)
    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(x_train, y_train),
        batch_size=batch_size, shuffle=True, drop_last=True)


    
    model = MDN(n_hidden=neurons, n_gaussians=2,n_hidden_layers=layers1,bn=bnorm, dout=dout, dout_value=dropout_rate)
    model.to(device)
    
    PATH = "checkpoint/model-946L.pt" 
    checkpoint = torch.load(PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])



    ax=[]
    ax_test=[]
    final=[]
    for epoch in range(n_epochs):
        model.train()
        for j, (x, y) in enumerate(train_loader):
            x, y = Variable(x).to(device), Variable(y).to(device)
            pi_variable, sigma_variable, mu_variable = model(x)
            loss = mdn_loss_fn(pi_variable, sigma_variable, mu_variable, y)
            optimizer.zero_grad()
            loss.backward()
            nn_utils.clip_grad_norm_(model.parameters(), max_norm=0.5)  
            optimizer.step()
            
        model.eval()
        for j, (x, y) in enumerate(test_loader):
            x, y = Variable(x).to(device), Variable(y).to(device)
            optimizer.zero_grad()
            pi_variable, sigma_variable, mu_variable = model(x)
            loss_test = mdn_loss_fn(pi_variable, sigma_variable, mu_variable, y)
        if epoch %10==0:
            
            print(f"epoch: {epoch}, " + f"Loss: {loss.data:.2f}")


        PATH="checkpoint/model-"+str(epoch)+".pt"
        torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                }, PATH)
        ax_test=np.append(ax_test,loss_test.data)   
        ax=np.append(ax,loss.data)

        d=np.zeros([1,3])
        d[0,0]=epoch
        d[0,1]=loss.data
        d[0,2]=loss_test.data
        with open("res_fold1.txt","ab") as file:
            np.savetxt(file,d,fmt='%.5f %.5f %.5f')

        final=np.append(final,loss_test.data)
        
    c=np.arange(0,ax.shape[0],1).reshape(-1,1)
    ax=ax.reshape(-1,1)
    ax_test=ax_test.reshape(-1,1)
    c=np.hstack([c,ax,ax_test])
    with open('res.npy','wb') as f:
        np.save(f,c)
    return np.min(final)
   
    
    
    
data_x = np.load('xlo_oversampling.npy')[:, :]
data_y = np.load('ylo_oversampling.npy')[:,:]
nn_cl_bo2(dropout=0,dropout_rate=0.21738,normalization=0,batch_size=486,layers1=6,neurons=91,learning_rate=0.00168)
